# Final Results Table and Paper-Ready Figures

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

Run this notebook **last**, after all training and benchmark notebooks  
have been completed for every horizon.

This notebook produces:
1. A unified cross-model, cross-horizon results table (CSV + LaTeX)
2. Filtered (clean subset) metrics for NiOA-DRNN alongside standard metrics
3. Best-vs-second-best delta analysis
4. Per-horizon improvement percentage over each baseline
5. All publication-quality plots required for the paper

### v2 addition
Section 2 loads `metrics_filtered.json` from each NiOA-DRNN experiment and  
reports the clean-subset metrics alongside the standard full-test-set metrics.  
Both are included in the final tables for full transparency.


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'core')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config import (
    FORECAST_HORIZONS, BENCHMARK_DIR, NIOA_RESULTS_DIR, RESULTS_DIR
)

ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)

sns.set(style='whitegrid', palette='tab10', font_scale=1.1)
print(f'Project root : {PROJECT_ROOT}')
print('Final results notebook ready.')

---
## 1 · Load All Saved Metrics (Standard — Full Test Set)

In [ ]:
def _load_nioa_metrics(nioa_root, horizon):
    """Find the most recent NiOA-DRNN experiment for the given horizon."""
    if not os.path.isdir(nioa_root):
        return None
    _cands = [d for d in os.listdir(nioa_root) if d.startswith(f'k{horizon}_')]
    if not _cands:
        return None
    _latest = sorted(_cands)[-1]
    _path   = os.path.join(nioa_root, _latest, 'metrics.json')
    if not os.path.isfile(_path):
        return None
    with open(_path) as _f:
        return json.load(_f), os.path.join(nioa_root, _latest)


all_rows = []

for h in FORECAST_HORIZONS:
    # ── Benchmark models (from results/benchmark/horizon_k/) ─────────────────
    _hdir = os.path.join(BENCHMARK_DIR, f'horizon_{h}')
    if os.path.isdir(_hdir):
        for _mname in sorted(os.listdir(_hdir)):
            _mp = os.path.join(_hdir, _mname, 'metrics.json')
            if os.path.isfile(_mp):
                with open(_mp) as _f:
                    _m = json.load(_f)
                all_rows.append({'Model': _mname, 'Horizon_s': h, **_m})

    # ── Proposed NiOA-DRNN ────────────────────────────────────────────────────
    _result = _load_nioa_metrics(NIOA_RESULTS_DIR, h)
    if _result:
        _nioa_m, _exp_dir = _result
        all_rows.append({
            'Model'    : 'nioa_drnn',
            'Horizon_s': h,
            'MAE'      : _nioa_m.get('MAE',   _nioa_m.get('mae',   float('nan'))),
            'RMSE'     : _nioa_m.get('RMSE',  _nioa_m.get('rmse',  float('nan'))),
            'R2'       : _nioa_m.get('R2',    _nioa_m.get('r2',    float('nan'))),
            'sMAPE'    : _nioa_m.get('sMAPE', _nioa_m.get('smape', float('nan'))),
        })

results = pd.DataFrame(all_rows)

MODEL_LABELS = {
    'linear_regression': 'Linear Regression',
    'svr'              : 'SVR',
    'xgboost'          : 'XGBoost',
    'mlp'              : 'MLP',
    'arima'            : 'ARIMA',
    'vanilla_lstm'     : 'Vanilla LSTM',
    'cnn_lstm'         : 'CNN-LSTM',
    'drnn_optuna'      : 'DRNN + Optuna',
    'nioa_drnn'        : 'DRNN + NiOA (Proposed)',
}
results['Model_Label'] = results['Model'].map(MODEL_LABELS).fillna(results['Model'])

print(f'Loaded: {results["Model"].nunique()} models  ×  '
      f'{results["Horizon_s"].nunique()} horizons')
print(results[['Model_Label', 'Horizon_s', 'MAE', 'RMSE', 'R2', 'sMAPE']]
      .to_string(index=False, float_format='{:.4f}'.format))

---
## 2 · NiOA-DRNN Filtered Metrics (Physical Plausibility Subset)

The `metrics_filtered.json` files were produced by `evaluate_filtered.py`  
and contain two metric sets:
- `metrics_full_test`   — standard metrics on the complete test set (includes artefacts)
- `metrics_clean_subset` — metrics after removing physically implausible samples

Both sets are loaded and reported here for full transparency.


In [ ]:
filtered_rows = []

for h in FORECAST_HORIZONS:
    _result = _load_nioa_metrics(NIOA_RESULTS_DIR, h)
    if not _result:
        continue
    _, _exp_dir = _result
    _fpath = os.path.join(_exp_dir, 'metrics_filtered.json')
    if not os.path.isfile(_fpath):
        print(f'  k={h}: metrics_filtered.json not found — run evaluate_filtered.py first.')
        continue
    with open(_fpath) as _f:
        _fdata = json.load(_f)

    filtered_rows.append({
        'Horizon_s'          : h,
        'n_test_total'       : _fdata['n_test_total'],
        'n_clean'            : _fdata['n_test_clean'],
        'pct_artefacts'      : _fdata['pct_artefacts'],
        'max_plausible_kwh'  : _fdata['max_plausible_kwh'],
        # Full test set
        'MAE_full'           : _fdata['metrics_full_test']['MAE'],
        'RMSE_full'          : _fdata['metrics_full_test']['RMSE'],
        'R2_full'            : _fdata['metrics_full_test']['R2'],
        'sMAPE_full'         : _fdata['metrics_full_test']['sMAPE'],
        # Clean subset
        'MAE_clean'          : _fdata['metrics_clean_subset']['MAE'],
        'RMSE_clean'         : _fdata['metrics_clean_subset']['RMSE'],
        'R2_clean'           : _fdata['metrics_clean_subset']['R2'],
        'sMAPE_clean'        : _fdata['metrics_clean_subset']['sMAPE'],
    })

if filtered_rows:
    df_filtered = pd.DataFrame(filtered_rows)
    print('NiOA-DRNN — Filtered vs Full Test Set Metrics')
    print('=' * 80)
    print(df_filtered.to_string(index=False, float_format='{:.6f}'.format))
    print('=' * 80)
    _fcsv = os.path.join(ANALYSIS_DIR, 'nioa_drnn_filtered_metrics.csv')
    df_filtered.to_csv(_fcsv, index=False)
    print(f'\nSaved → {_fcsv}')
else:
    print('No filtered metrics found.  Run evaluate_filtered.py for each horizon.')

---
## 3 · Paper Table — Full Metrics Pivot (all models)

In [ ]:
MODEL_ORDER = [
    'Linear Regression', 'SVR', 'XGBoost', 'MLP', 'ARIMA',
    'Vanilla LSTM', 'CNN-LSTM', 'DRNN + Optuna', 'DRNN + NiOA (Proposed)'
]

for _metric in ['MAE', 'RMSE', 'R2', 'sMAPE']:
    _pivot = (
        results
        .pivot(index='Model_Label', columns='Horizon_s', values=_metric)
        .reindex([m for m in MODEL_ORDER if m in results['Model_Label'].values])
    )
    _pivot.columns = [f'k={h}s' for h in _pivot.columns]

    print(f'\n{"="*55}\n{_metric} Comparison — Full Test Set\n{"="*55}')
    print(_pivot.to_string(float_format='{:.4f}'.format))

    _pivot.to_csv(os.path.join(ANALYSIS_DIR, f'table_{_metric.lower()}.csv'))

    _latex = _pivot.to_latex(
        float_format='{:.4f}'.format,
        bold_rows=False,
        caption=f'{_metric} comparison across models and forecast horizons (full test set).',
        label=f'tab:{_metric.lower()}_comparison',
    )
    with open(os.path.join(ANALYSIS_DIR, f'table_{_metric.lower()}.tex'), 'w') as _f:
        _f.write(_latex)

print(f'\nAll tables saved to : {ANALYSIS_DIR}')

---
## 4 · NiOA Improvement Over Each Baseline (%)

In [ ]:
_nioa = results[results['Model'] == 'nioa_drnn'].set_index('Horizon_s')
_imp_rows = []

for _, _row in results[results['Model'] != 'nioa_drnn'].iterrows():
    _h = _row['Horizon_s']
    if _h not in _nioa.index:
        continue
    _base_mae  = _row['MAE']
    _nioa_mae  = _nioa.loc[_h, 'MAE']
    _base_rmse = _row['RMSE']
    _nioa_rmse = _nioa.loc[_h, 'RMSE']
    _pct_mae   = (_base_mae  - _nioa_mae)  / _base_mae  * 100 if _base_mae  > 0 else float('nan')
    _pct_rmse  = (_base_rmse - _nioa_rmse) / _base_rmse * 100 if _base_rmse > 0 else float('nan')
    _imp_rows.append({
        'Baseline'          : _row['Model_Label'],
        'Horizon'           : _h,
        'MAE_improvement_%' : round(_pct_mae, 2),
        'RMSE_improvement_%': round(_pct_rmse, 2),
    })

_imp_df = pd.DataFrame(_imp_rows)
if not _imp_df.empty:
    _piv = _imp_df.pivot(index='Baseline', columns='Horizon', values='MAE_improvement_%')
    print('NiOA-DRNN MAE improvement (%) over each baseline — full test set:')
    print(_piv.to_string(float_format='{:.2f}'.format))
    _piv.to_csv(os.path.join(ANALYSIS_DIR, 'nioa_improvement_pct.csv'))
else:
    print('No NiOA results found yet — run notebook 01 first.')

---
## 5 · Publication Figures

In [ ]:
if results.empty:
    print('No results to plot.')
else:
    _avail_h = sorted(results['Horizon_s'].unique())

    # A. MAE vs Horizon
    _fig, _ax = plt.subplots(figsize=(11, 5))
    for _label, _grp in results.groupby('Model_Label'):
        _is_prop = _label == 'DRNN + NiOA (Proposed)'
        _ax.plot(
            _grp['Horizon_s'].values, _grp['MAE'].values,
            marker='o' if _is_prop else 's',
            lw=2.5 if _is_prop else 1.5,
            ls='-'  if _is_prop else '--',
            ms=7, label=_label
        )
    _ax.set_xscale('log')
    _ax.set_xticks(_avail_h)
    _ax.set_xticklabels([str(h) for h in _avail_h])
    _ax.set_xlabel('Forecast Horizon k (seconds)', fontsize=12)
    _ax.set_ylabel('MAE (kWh) — Full Test Set', fontsize=12)
    _ax.set_title('MAE vs Forecast Horizon — All Models', fontsize=13)
    _ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, 'mae_vs_horizon_all.png'),
                dpi=200, bbox_inches='tight')
    plt.close('all')

    # B. R² vs Horizon
    _fig, _ax = plt.subplots(figsize=(11, 5))
    for _label, _grp in results.groupby('Model_Label'):
        _is_prop = _label == 'DRNN + NiOA (Proposed)'
        _ax.plot(
            _grp['Horizon_s'].values, _grp['R2'].values,
            marker='o', lw=2.5 if _is_prop else 1.5,
            ls='-' if _is_prop else '--', ms=7, label=_label
        )
    _ax.axhline(0, color='black', ls=':', lw=0.9)
    _ax.set_xscale('log')
    _ax.set_xticks(_avail_h)
    _ax.set_xticklabels([str(h) for h in _avail_h])
    _ax.set_xlabel('Forecast Horizon k (seconds)', fontsize=12)
    _ax.set_ylabel('R²', fontsize=12)
    _ax.set_title('R² vs Forecast Horizon — All Models', fontsize=13)
    _ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, 'r2_vs_horizon_all.png'),
                dpi=200, bbox_inches='tight')
    plt.close('all')

    # C. Per-horizon MAE bar
    _nh = len(_avail_h)
    _fig, _axes = plt.subplots(1, _nh, figsize=(5 * _nh, 5), sharey=False)
    if _nh == 1:
        _axes = [_axes]
    for _ax, _h in zip(_axes, _avail_h):
        _sub = results[results['Horizon_s'] == _h].sort_values('MAE')
        _colours = [
            'darkorange' if m == 'DRNN + NiOA (Proposed)' else 'steelblue'
            for m in _sub['Model_Label']
        ]
        _ax.barh(_sub['Model_Label'], _sub['MAE'], color=_colours)
        _ax.set_title(f'k = {_h} s')
        _ax.set_xlabel('MAE (kWh)')
        _ax.invert_yaxis()
    plt.suptitle(
        'MAE Comparison — All Models  (orange = proposed NiOA-DRNN)',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, 'mae_bar_per_horizon.png'),
                dpi=200, bbox_inches='tight')
    plt.close('all')

    # D. NiOA-DRNN clean vs full MAE across horizons (if filtered data available)
    if filtered_rows:
        _fig, _ax = plt.subplots(figsize=(9, 5))
        _ax.plot(
            df_filtered['Horizon_s'], df_filtered['MAE_full'],
            marker='o', lw=2, color='coral', label='Full test set (with artefacts)'
        )
        _ax.plot(
            df_filtered['Horizon_s'], df_filtered['MAE_clean'],
            marker='s', lw=2, color='steelblue', label='Clean subset (artefacts removed)'
        )
        _ax.set_xscale('log')
        _ax.set_xticks(df_filtered['Horizon_s'].tolist())
        _ax.set_xticklabels([str(h) for h in df_filtered['Horizon_s']])
        _ax.set_xlabel('Forecast Horizon k (seconds)', fontsize=12)
        _ax.set_ylabel('MAE (kWh)', fontsize=12)
        _ax.set_title('NiOA-DRNN — Full vs Clean Subset MAE', fontsize=13)
        _ax.legend(fontsize=10)
        plt.tight_layout()
        plt.savefig(os.path.join(ANALYSIS_DIR, 'nioa_full_vs_clean_mae.png'),
                    dpi=200, bbox_inches='tight')
        plt.close('all')

        # sMAPE trend (clean subset) — shows improvement with horizon
        _fig, _ax = plt.subplots(figsize=(9, 5))
        _ax.plot(
            df_filtered['Horizon_s'], df_filtered['sMAPE_clean'],
            marker='o', lw=2.5, color='darkorange'
        )
        _ax.set_xscale('log')
        _ax.set_xticks(df_filtered['Horizon_s'].tolist())
        _ax.set_xticklabels([str(h) for h in df_filtered['Horizon_s']])
        _ax.set_xlabel('Forecast Horizon k (seconds)', fontsize=12)
        _ax.set_ylabel('sMAPE (%) — Clean Subset', fontsize=12)
        _ax.set_title('NiOA-DRNN — sMAPE Trend Across Horizons (Clean Subset)', fontsize=13)
        for _x, _y in zip(df_filtered['Horizon_s'], df_filtered['sMAPE_clean']):
            _ax.annotate(f'{_y:.1f}%', (_x, _y),
                         textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(ANALYSIS_DIR, 'nioa_smape_clean_trend.png'),
                    dpi=200, bbox_inches='tight')
        plt.close('all')

    print(f'All publication figures saved to : {ANALYSIS_DIR}')

---
## 6 · Final Printout — Complete Summary

In [ ]:
print('\n' + '='*70)
print('  COMPLETE RESULTS SUMMARY')
print('='*70)

if not results.empty:
    print('\n--- Standard Metrics (Full Test Set) ---')
    print(results[['Model_Label', 'Horizon_s', 'MAE', 'RMSE', 'R2', 'sMAPE']]
          .sort_values(['Horizon_s', 'MAE'])
          .to_string(index=False, float_format='{:.4f}'.format))

if filtered_rows:
    print('\n--- NiOA-DRNN Filtered Metrics (Physical Plausibility Subset) ---')
    print(df_filtered[[
        'Horizon_s', 'n_test_total', 'n_clean', 'pct_artefacts',
        'MAE_full', 'MAE_clean', 'R2_full', 'R2_clean', 'sMAPE_full', 'sMAPE_clean'
    ]].to_string(index=False, float_format='{:.4f}'.format))

print('\n' + '='*70)